In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 58 samples and 12013 columns.
First few rows of the data:


,overall survival (months),vital status,sample_title,tissue,unique sample id,patient id,grade,batch,idh status prediction,age,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,115.733333,1,Sample_S-10-33895_S22_L003merged,Glioma,Sample_S-10-33895,1,HGG,1,Mutant,22.0,...,6.416128,4.085363,5.955792,6.335819,4.288661,5.358118,6.942101,6.587702,5.318951,6.187649
1,26.833333,1,Sample_R378-14-133900_S12_L002,Glioma,Sample_14-133900,10,HGG,2,WT,39.0,...,6.874357,3.695918,5.518125,6.827268,4.515771,5.706525,7.234463,6.714490,4.857237,6.149874
2,34.433333,1,Sample_S-12-53702_S22_L003merged,Glioma,Sample_S-12-53702,103,HGG,1,WT,59.0,...,6.856569,4.238486,6.308461,7.027931,4.694066,5.504444,7.328217,7.096958,5.321711,6.411636
3,182.466667,1,Sample_S-13-16523_S22_L003merged,Glioma,Sample_S-13-16523,104,HGG,1,Mutant,31.0,...,7.261454,4.524313,6.533104,7.372353,4.526102,5.855307,7.544500,7.244535,5.657381,6.958492
4,8.866667,0,Sample_R378-15-49530_S13_L002,Glioma,Sample_15-49530,11,HGG,2,Mutant,49.0,...,5.670679,3.080187,4.963198,5.792980,3.826287,5.216755,6.434277,6.151793,4.775085,5.818796


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for overall survival (months) or vital status: 1
Number of samples with complete data for both variables: 57
Number of censored samples (event_var == 0.0): 22
Number of events (event_var == 1.0): 35
Censored ratio (among complete): 22 / 57 = 0.386 (38.6%)
Skipping stats for PFS: columns not in preprocessed table ('progress free survival (months)', 'unknown').
Updated stats for 1 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE184941/description.json
